# Quiz Generation from Student Notes

**Authors:** Naseeha Masub (ID: ) & Calvin Chau (ID: 115407976)  
**Course:** CSE 354

---

# Generating Multiple-Choice Quizzes from Student Notes

## Project overview

Our application allows a student to paste their lecture notes and generate
a multiple-choice practice quiz. The student can enter paragraphs, headings,
or bullet points without organizing the notes into a spreadsheet.

The notebook divides the notes into passages, identifies possible answer
terms, and generates questions with FLAN-T5. Each question has one intended
correct answer and three generated distractors.

## Research question

How does the decoding strategy affect the accuracy, relevance, distractor
quality, and diversity of questions generated from the same student notes?

We compare greedy decoding, beam search, and sampling at two temperatures.
We use the same model, passages, target answers, and prompts for all four
setups. We manually evaluate the outputs to determine which questions would
actually be useful to a student.

In [ ]:
%pip -q install "transformers==4.57.1" sentencepiece ipywidgets

: 

In [ ]:
import hashlib
import json
import random
import re
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import torch

from google.colab import drive, output
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed

drive.mount("/content/drive")
output.enable_custom_widget_manager()

project_dir = Path("/content/drive/MyDrive/NLP_Quiz_Project")
project_dir.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print("Project folder:", project_dir)

## 1. Entering student notes

Paste the notes into the text box below. The notebook preserves the original
text and automatically creates passage IDs. The student does not need to
identify keywords or supply the answers beforehand.

For the pilot, we can use a short lecture excerpt to inspect the generated
questions. For the final experiment, we will use notes that were not used
while revising our prompts.

In [ ]:
last_notes_path = project_dir / "last_notes.txt"

notes_box = widgets.Textarea(
    value=last_notes_path.read_text(encoding="utf-8") if last_notes_path.exists() else "",
    placeholder="Paste your lecture notes here...",
    description="Notes:",
    layout=widgets.Layout(width="95%", height="320px")
)

display(notes_box)

In [ ]:
raw_notes = notes_box.value.strip()

if not raw_notes:
    raise ValueError("Paste your notes into the text box in Cell 5 first.")

if not re.search(r"[A-Za-z]", raw_notes):
    raise ValueError("Enter notes containing English text.")

# Keep a separate copy of each distinct input.
notes_id = hashlib.sha256(raw_notes.encode()).hexdigest()[:12]
notes_dir = project_dir / f"notes_{notes_id}"
notes_dir.mkdir(exist_ok=True)

(notes_dir / "original_notes.txt").write_text(raw_notes, encoding="utf-8")
last_notes_path.write_text(raw_notes, encoding="utf-8")


def split_notes(text, max_words=100):
    passages = []

    # A blank line usually indicates that the student changed topics.
    for paragraph in re.split(r"\n\s*\n", text.strip()):
        current = []

        # Within a paragraph, keep sentences or bullet lines together when possible.
        for section in re.split(r"\n+|(?<=[.!?])\s+", paragraph):
            words = section.split()
            if not words:
                continue

            if current and len(current) + len(words) > max_words:
                passages.append(" ".join(current))
                current = []

            # Long pasted blocks may have no punctuation.
            while len(words) > max_words:
                passages.append(" ".join(words[:max_words]))
                words = words[max_words:]

            current.extend(words)

        if current:
            passages.append(" ".join(current))

    return passages


passages = split_notes(raw_notes)

notes = pd.DataFrame({
    "passage_id": [f"P{i:02d}" for i in range(1, len(passages) + 1)],
    "passage": passages
})

print(f"Created {len(notes)} passage(s).")
display(notes)

## 2. Selecting possible answers with TF-IDF

We use TF-IDF to rank terms from the pasted notes. We consider individual
words and two-word phrases, which allows the notebook to select terms such
as "recall" or "beam search."

For each passage, we choose up to two distinct target answers. All four
decoding setups receive the same targets. We display them before generation
because a term can receive a high score without being a useful quiz answer.

In [ ]:
# Remove common English stop words through the vectorizer. Also exclude
# standalone headings and instructions that commonly appear in lecture notes.
heading_words = {
    "lecture", "lectures", "chapter", "slide", "slides", "example",
    "examples", "definition", "definitions", "overview", "notes"
}

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-Z][\w-]*\b"
)

try:
    matrix = vectorizer.fit_transform(notes["passage"])
except ValueError as error:
    raise ValueError("No usable terms were found. Paste a longer excerpt.") from error

terms = vectorizer.get_feature_names_out()
targets = []

for row_number, note in enumerate(notes.itertuples(index=False)):
    scores = matrix.getrow(row_number)

    ranked = sorted(
        zip(scores.indices, scores.data),
        key=lambda pair: (-pair[1], terms[pair[0]])
    )

    selected = []

    for term_index, score in ranked:
        term = terms[term_index]
        words = set(term.split())

        if words & heading_words or len(term) < 2:
            continue

        # Stop-word removal can create a bigram from nonadjacent words.
        # Only keep answer phrases that actually appear in the passage.
        match = re.search(
            rf"(?<!\w){re.escape(term)}(?!\w)",
            note.passage,
            flags=re.IGNORECASE
        )
        if match is None:
            continue

        # Avoid selecting both "search" and "beam search" for one passage.
        if any(words <= previous or previous <= words for previous in selected):
            continue

        selected.append(words)
        targets.append({
            "passage_id": note.passage_id,
            "passage": note.passage,
            "target": match.group(),
            "tfidf": round(float(score), 4)
        })

        if len(selected) == 2:
            break

target_df = pd.DataFrame(targets)

if target_df.empty:
    raise ValueError("No target answers were found. Try a more detailed excerpt.")

print(f"Selected {len(target_df)} target answer(s).")
display(target_df[["passage_id", "target", "tfidf"]])

## 3. Generating questions with FLAN-T5

We use FLAN-T5-base with four decoding setups:

1. Greedy decoding selects the most likely next token at each step.
2. Beam search keeps four candidate sequences as generation proceeds.
3. Lower-temperature sampling uses a temperature of 0.7 and top-p of 0.9.
4. Higher-temperature sampling uses a temperature of 1.0 and top-p of 0.9.

The model first writes a question for a target answer. It then generates
three distractors. The same decoding setup is used for both tasks.

This experiment changes generation settings. The pretrained model weights
stay fixed throughout the comparison.

In [ ]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

setups = {
    "greedy": {
        "do_sample": False,
        "num_beams": 1
    },
    "beam_4": {
        "do_sample": False,
        "num_beams": 4
    },
    "sample_0.7": {
        "do_sample": True,
        "num_beams": 1,
        "top_k": 0,
        "top_p": 0.9,
        "temperature": 0.7
    },
    "sample_1.0": {
        "do_sample": True,
        "num_beams": 1,
        "top_k": 0,
        "top_p": 0.9,
        "temperature": 1.0
    }
}

question_prompt = """Write one short practice question using these notes.
The correct answer must be "{target}".
Do not reveal the answer in the question. Output only the question.

Notes: {passage}
Question:"""

choice_prompt = """Write one plausible but incorrect answer to the question.
Use a short answer of the same general type as the correct answer.
Avoid synonyms of the correct answer. Avoid repeating earlier choices.
Output only the new incorrect choice.

Notes: {passage}
Question: {question}
Correct answer: {target}
Earlier incorrect choices: {previous}
New incorrect choice:"""


@torch.inference_mode()
def generate_text(prompt, settings, limit):
    inputs = tokenizer(prompt, return_tensors="pt")

    if inputs["input_ids"].shape[1] > 512:
        raise ValueError("Prompt exceeds 512 tokens. Reduce max_words in Cell 6.")

    inputs = inputs.to(device)
    generated = model.generate(**inputs, **settings, max_new_tokens=limit)
    return tokenizer.decode(generated[0], skip_special_tokens=True).strip()


def generate_question(passage, target, settings):
    prompt = question_prompt.format(passage=passage, target=target)
    question = generate_text(prompt, settings, limit=48)

    distractors = []

    for _ in range(3):
        prompt = choice_prompt.format(
            passage=passage,
            question=question,
            target=target,
            previous=", ".join(distractors) or "none"
        )
        distractors.append(generate_text(prompt, settings, limit=16))

    return question, distractors


print("Loaded", model_name, "on", device)

In [ ]:
base_seed = 42

experiment = {
    "model": model_name,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "setups": setups,
    "seed": base_seed,
    "targets": target_df[["passage_id", "passage", "target"]].to_dict("records"),
    "question_prompt": question_prompt,
    "choice_prompt": choice_prompt,
    "question_token_limit": 48,
    "choice_token_limit": 16
}

experiment_text = json.dumps(experiment, sort_keys=True)
run_id = hashlib.sha256(experiment_text.encode()).hexdigest()[:12]

run_dir = notes_dir / f"run_{run_id}"
run_dir.mkdir(exist_ok=True)
(run_dir / "experiment.json").write_text(experiment_text, encoding="utf-8")

results = []
expected = len(target_df) * len(setups)
print(f"Expected generation attempts: {expected}")

for target_index, row in enumerate(target_df.itertuples(index=False)):
    for setup_name, settings in setups.items():
        item_id = f"{row.passage_id}_{target_index}_{setup_name}"
        path = run_dir / f"{item_id}.json"

        if path.exists():
            item = json.loads(path.read_text(encoding="utf-8"))
        else:
            set_seed(base_seed + target_index)

            question, distractors = generate_question(
                row.passage, row.target, settings
            )

            item = {
                "item_id": item_id,
                "passage_id": row.passage_id,
                "passage": row.passage,
                "target": row.target,
                "setup": setup_name,
                "question": question,
                "distractors": distractors
            }

            temporary = path.with_suffix(".tmp")
            temporary.write_text(json.dumps(item, indent=2), encoding="utf-8")
            temporary.replace(path)
            print("Generated:", item_id)

        results.append(item)

results_df = pd.DataFrame(results)

print("\nCompleted attempts per setup:")
print(results_df.groupby("setup").size())
print("\nExperiment folder:", run_dir)

In [ ]:
first_target = target_df.iloc[0]

comparison = results_df[
    (results_df["passage_id"] == first_target["passage_id"]) &
    (results_df["target"] == first_target["target"])
]

print("SOURCE PASSAGE")
print(first_target["passage"])
print("\nTARGET ANSWER:", first_target["target"])

for item in comparison.itertuples(index=False):
    print(f"\n[{item.setup}]")
    print("Question:", item.question)
    print("Distractors:", " | ".join(item.distractors))

In [ ]:
quiz_setup = "greedy"  # greedy, beam_4, sample_0.7, or sample_1.0
take_quiz_now = True  # Set to False to export without answering in Colab.

if quiz_setup not in setups:
    raise ValueError("Choose one of the four setup names from Cell 10.")


def shuffled_choices(item, rng):
    choices = [(item["target"], True)]
    choices += [(text, False) for text in item["distractors"]]
    rng.shuffle(choices)

    options = {}
    for letter, (text, is_key) in zip("ABCD", choices):
        options[letter] = text
        if is_key:
            key = letter

    return options, key


quiz_items = [item for item in results if item["setup"] == quiz_setup]
quiz_rng = random.Random(42)
quiz_lines = []
key_lines = []
score = 0

for number, item in enumerate(quiz_items, start=1):
    options, key = shuffled_choices(item, quiz_rng)

    lines = [f"{number}. {item['question']}"]
    lines += [f"{letter}. {text}" for letter, text in options.items()]
    question_block = "\n".join(lines)

    quiz_lines.append(question_block)
    key_lines.append(f"{number}. {key}. {item['target']}")

    if take_quiz_now:
        print("\n" + question_block)
        answer = input("Your choice (A/B/C/D): ").strip().upper()
        while answer not in options:
            answer = input("Enter A, B, C, or D: ").strip().upper()

        score += int(answer == key)
        print("Intended answer:", key, "-", item["target"])
        print("Source:", item["passage"])

if take_quiz_now:
    print(f"\nYou matched the intended key on {score}/{len(quiz_items)} questions.")

quiz_path = run_dir / f"quiz_{quiz_setup}.txt"
student_key_path = run_dir / f"quiz_{quiz_setup}_answers.txt"

quiz_path.write_text("\n\n".join(quiz_lines), encoding="utf-8")
student_key_path.write_text("\n".join(key_lines), encoding="utf-8")

print("Quiz:", quiz_path)
print("Answer key:", student_key_path)

## 4. Manual evaluation

We shuffle questions from all four setups and hide the setup names during
review. Reviewers read each source passage and answer each question before
seeing the intended answer key.

For individual questions, we evaluate:

- Answer accuracy: Does exactly one choice answer the question, and does it
  match the intended key?
- Relevance: Does the question test meaningful information from the passage?
- Distractor quality: How many choices are distinct, clearly incorrect,
  and plausible enough to tempt a student?

For passages with two selected targets, we also evaluate diversity. We view
the two questions from the same setup together and check whether they test
two different facts supported by the passage.

In [ ]:
review_rng = random.Random(100)
review_items = results.copy()
review_rng.shuffle(review_items)

review_rows = []
key_rows = []

for number, item in enumerate(review_items, start=1):
    review_id = f"Q{number:03d}"
    options, key = shuffled_choices(item, review_rng)

    review_rows.append({
        "review_id": review_id,
        "passage": item["passage"],
        "question": item["question"],
        **options,
        "reviewer_answer": "",
        "relevant": "",
        "plausible_distractors": "",
        "comment": ""
    })

    key_rows.append({
        "review_id": review_id,
        "item_id": item["item_id"],
        "setup": item["setup"],
        "key_letter": key
    })

review_sheet = pd.DataFrame(review_rows)
answer_key = pd.DataFrame(key_rows)

# Build a separate blind sheet for comparing pairs of questions.
pairs = []
for (passage_id, setup), group in results_df.groupby(["passage_id", "setup"]):
    if len(group) == 2:
        pairs.append({
            "setup": setup,
            "passage": group["passage"].iloc[0],
            "question_1": group["question"].iloc[0],
            "question_2": group["question"].iloc[1]
        })

review_rng.shuffle(pairs)
pair_rows = []
pair_keys = []

for number, pair in enumerate(pairs, start=1):
    pair_id = f"D{number:03d}"
    pair_rows.append({
        "pair_id": pair_id,
        "passage": pair["passage"],
        "question_1": pair["question_1"],
        "question_2": pair["question_2"],
        "distinct_facts": ""
    })
    pair_keys.append({"pair_id": pair_id, "setup": pair["setup"]})


def save_review_template(frame, filename):
    path = run_dir / filename
    if not path.exists():
        frame.to_csv(path, index=False)
    print(path)


save_review_template(review_sheet, "review_sheet.csv")
answer_key.to_csv(run_dir / "review_key.csv", index=False)

if pairs:
    save_review_template(pd.DataFrame(pair_rows), "diversity_review.csv")
    pd.DataFrame(pair_keys).to_csv(run_dir / "diversity_key.csv", index=False)
else:
    print("No passage had two targets, so there are no diversity pairs to review.")

print(f"\nQuestions to review: {len(review_rows)}")
print(f"Pairs to review: {len(pair_rows)}")

## Reviewer instructions

Open review_sheet.csv in Google Sheets. Leave the review key closed.

For every question, fill in these columns:

- reviewer_answer: Enter A, B, C, or D when exactly one option answers the
  question. Enter MULTIPLE when several options work. Enter NONE when no
  option works or the passage does not support an answer.
- relevant: Enter 1 if the question tests meaningful information from the
  passage. Otherwise enter 0.
- plausible_distractors: Enter a whole number from 0 to 3. Count distinct
  options that are incorrect but believable. A repeated option earns no
  extra credit.
- comment: Briefly explain a strength or problem. Record examples we may
  want to use in the presentation.

Download the completed sheet as review_scored.csv and put it in the same
experiment folder as review_sheet.csv.

If diversity_review.csv was created, review each pair. Enter 1 in
distinct_facts when the two questions test different facts supported by
the passage. Enter 0 if they repeat the same fact or either question is
unusable. Download the completed file as diversity_scored.csv and put it
in the same experiment folder.

In [ ]:
# After a fresh session, uncomment this and paste your actual folder path:
# run_dir = Path("/content/drive/MyDrive/NLP_Quiz_Project/notes_.../run_...")

def read_scored_file(scored_name, key_name, id_column):
    scored = pd.read_csv(run_dir / scored_name, keep_default_na=False)
    key = pd.read_csv(run_dir / key_name, keep_default_na=False)

    if scored[id_column].duplicated().any() or key[id_column].duplicated().any():
        raise ValueError(f"Duplicate IDs found in {scored_name} or {key_name}.")

    if set(scored[id_column]) != set(key[id_column]):
        raise ValueError(f"The IDs in {scored_name} do not match this experiment.")

    return scored.merge(key, on=id_column, validate="one_to_one")


def numeric_scores(frame, column, allowed):
    values = pd.to_numeric(frame[column], errors="coerce")
    if not values.isin(allowed).all():
        raise ValueError(f"Fill every {column} cell with one of {allowed}.")
    return values.astype(int)


scored_path = run_dir / "review_scored.csv"

if not scored_path.exists():
    print("Waiting for the completed review_scored.csv in:", run_dir)
else:
    graded = read_scored_file("review_scored.csv", "review_key.csv", "review_id")
    graded["reviewer_answer"] = graded["reviewer_answer"].str.strip().str.upper()

    allowed_answers = ["A", "B", "C", "D", "MULTIPLE", "NONE"]
    if not graded["reviewer_answer"].isin(allowed_answers).all():
        raise ValueError("Check reviewer_answer for blanks or invalid entries.")

    graded["relevant"] = numeric_scores(graded, "relevant", [0, 1])
    graded["plausible_distractors"] = numeric_scores(
        graded, "plausible_distractors", [0, 1, 2, 3]
    )
    graded["accurate"] = graded["reviewer_answer"] == graded["key_letter"]

    summary = graded.groupby("setup").agg(
        questions=("review_id", "count"),
        accurate_questions=("accurate", "sum"),
        accuracy_pct=("accurate", "mean"),
        relevance_pct=("relevant", "mean"),
        avg_plausible_distractors=("plausible_distractors", "mean")
    )

    summary[["accuracy_pct", "relevance_pct"]] *= 100

    if (run_dir / "diversity_scored.csv").exists():
        diversity = read_scored_file(
            "diversity_scored.csv", "diversity_key.csv", "pair_id"
        )
        diversity["distinct_facts"] = numeric_scores(diversity, "distinct_facts", [0, 1])

        pair_summary = diversity.groupby("setup").agg(
            pairs_reviewed=("pair_id", "count"),
            distinct_fact_pct=("distinct_facts", "mean")
        )
        pair_summary["distinct_fact_pct"] *= 100
        summary = summary.join(pair_summary)
    elif (run_dir / "diversity_key.csv").exists():
        print("Question scores are ready. Pair scores are pending diversity_scored.csv.")

    summary = summary.round(1)
    display(summary)
    summary.to_csv(run_dir / "setup_comparison.csv")